In [30]:
# !pip install h2o

# Loading Data

In [2]:
import pandas as pd
import numpy as np

In [31]:
# USE THIS WHEN RUNNING ON COLAB

import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

df_bat = pd.read_csv(path + '/ipl_bat_features.csv')
df_bowl = pd.read_csv(path + '/ipl_bowl_features.csv')
df_bat_fp = pd.read_csv(path + '/ipl_bat_fp.csv')
df_bowl_fp = pd.read_csv(path + '/ipl_bowl_fp.csv')

Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/data-dump/versions/7


In [5]:
df_bat.shape, df_bowl.shape

((24333, 34), (24444, 34))

# Data Preparation

In [6]:
df_bowl['date'] = pd.to_datetime(df_bowl['date'])
df_bat['date'] = pd.to_datetime(df_bat['date'])

In [7]:
# merging fantasy points in one single dataset

df_bat = df_bat.merge(
    df_bat_fp[['batting_fp', 'match_id', 'batter_id']],
    on = ['match_id', 'batter_id']
)

df_bowl = df_bowl.merge(
    df_bowl_fp[['bowling_fp', 'match_id', 'bowler_id']],
    on = ['match_id', 'bowler_id']
)

In [8]:
#sorting values by date and innings
df_bat = df_bat.sort_values(by = ['date', 'inns'])
df_bowl = df_bowl.sort_values(by = ['date', 'inns'])
df_bat.head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,venue,career_avg,career_rpm,...,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm,batting_fp
929,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,26184,Ajit Agarkar,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2340,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,227712,Ashok Dinda,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3295,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,37737,Brendon McCullum,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,298
4336,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,5766,David Hussey,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16
7649,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,236779,Ishant Sharma,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [9]:
df_bowl.head()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,venue,season,career_avg,...,ssn_eco,ssn_sr,ssn_wpm,ssn_opm,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm,bowling_fp
2384,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,6938,Ashley Noffke,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31
2811,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,26206,Balachandra Akhil,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3548,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,8291,Cameron White,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
7924,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,45789,Jacques Kallis,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
12047,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,44111,Mark Boucher,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [10]:
# Label Encoding categorical features

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
for column in ['bat_team', 'bowl_team', 'venue']:
    df_bat[column] = encoder.fit_transform(df_bat[column])
for column in ['bat_team', 'bowl_team', 'venue']:
    df_bowl[column] = encoder.fit_transform(df_bowl[column])

In [11]:
# dropping columns not required for training
df_bat.drop(columns=['date', 'match_id', 'batter'], inplace=True)
df_bowl.drop(columns=['date', 'match_id', 'bowler'], inplace=True)

In [12]:
df_bat.shape, df_bowl.shape

((24333, 32), (24664, 32))

# Train Test Val Split

- For full dataset (2008-24), use 23 as val set, and 24 as test set

In [13]:
# TRAIN VAL TEST SPLIT

df_bat_train = df_bat[df_bat['season'] < 2023]
df_bat_val = df_bat[df_bat['season'] >= 2023]
df_bowl_train = df_bowl[df_bowl['season'] < 2023]
df_bowl_val = df_bowl[df_bowl['season'] >= 2023]

In [14]:
df_bat_train.shape, df_bat_val.shape,df_bowl_train.shape, df_bowl_val.shape

((20878, 32), (3455, 32), (21208, 32), (3456, 32))

# Batting Model Training

In [15]:
import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import numpy as np

In [16]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.26" 2025-01-21; OpenJDK Runtime Environment (build 11.0.26+4-post-Ubuntu-1ubuntu122.04); OpenJDK 64-Bit Server VM (build 11.0.26+4-post-Ubuntu-1ubuntu122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.11/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp5ttufe2p
  JVM stdout: /tmp/tmp5ttufe2p/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmp5ttufe2p/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,07 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,5 days
H2O_cluster_name:,H2O_from_python_unknownUser_gl31t9
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.170 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [17]:
# Convert to H2OFrame
train = h2o.H2OFrame(df_bat_train)
valid = h2o.H2OFrame(df_bat_val)

# Define features and target variable
target = "batting_fp"   # Replace with your actual target column name
features = [col for col in train.columns if col != target]

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [18]:
# Run AutoML for 60 minutes
aml = H2OAutoML(
    max_runtime_secs=3600,   # 1 hour
    stopping_metric="RMSE",  # Since it's a regression problem
    seed=42
)
aml.train(x=features, y=target, training_frame=train, validation_frame=valid)

AutoML progress: |
19:02:11.927: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.

███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),5/6
# GBM base models (used / total),1/1
# XGBoost base models (used / total),1/1
# DeepLearning base models (used / total),1/1
# GLM base models (used / total),1/1
# DRF base models (used / total),1/2
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [19]:
lb = aml.leaderboard
lb

model_id,rmse,mse,mae,rmsle,mean_residual_deviance
StackedEnsemble_BestOfFamily_4_AutoML_1_20250401_190211,31.7204,1006.18,20.8146,nan,1006.18
StackedEnsemble_AllModels_4_AutoML_1_20250401_190211,31.7348,1007.1,20.8009,nan,1007.1
StackedEnsemble_AllModels_3_AutoML_1_20250401_190211,31.7424,1007.58,20.8108,nan,1007.58
GBM_grid_1_AutoML_1_20250401_190211_model_16,31.7493,1008.02,20.9243,nan,1008.02
StackedEnsemble_AllModels_2_AutoML_1_20250401_190211,31.7579,1008.56,20.8607,nan,1008.56
StackedEnsemble_AllModels_1_AutoML_1_20250401_190211,31.7616,1008.8,20.8611,nan,1008.8
StackedEnsemble_BestOfFamily_3_AutoML_1_20250401_190211,31.7621,1008.83,20.8815,nan,1008.83
StackedEnsemble_BestOfFamily_2_AutoML_1_20250401_190211,31.7786,1009.88,20.891,nan,1009.88
StackedEnsemble_BestOfFamily_1_AutoML_1_20250401_190211,31.7791,1009.91,20.9044,nan,1009.91
GBM_grid_1_AutoML_1_20250401_190211_model_14,31.8111,1011.94,21.0408,nan,1011.94


In [21]:
best_model = aml.leader  # Best model
performance = best_model.model_performance(valid)
print(performance)

ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 1211.5010561734316
RMSE: 34.806623739935354
MAE: 22.27743900958678
RMSLE: NaN
Mean Residual Deviance: 1211.5010561734316
R^2: 0.25893649544995534
Null degrees of freedom: 3454
Residual degrees of freedom: 3449
Null deviance: 5669707.68637745
Residual deviance: 4185736.1490792064
AIC: 34348.036513063475


# Bowling Model Training

In [24]:
# Convert to H2OFrame
train = h2o.H2OFrame(df_bowl_train)
valid = h2o.H2OFrame(df_bowl_val)

# Define features and target variable
target = "bowling_fp"   # Replace with your actual target column name
features = [col for col in train.columns if col != target]

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [25]:
# Run AutoML for 60 minutes
aml = H2OAutoML(
    max_runtime_secs=2500,   # 2500 seconds
    stopping_metric="RMSE",  # Since it's a regression problem
    seed=42
)
aml.train(x=features, y=target, training_frame=train, validation_frame=valid)

AutoML progress: |
20:10:13.219: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.

███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),6/6
# GBM base models (used / total),1/1
# XGBoost base models (used / total),1/1
# GLM base models (used / total),1/1
# DeepLearning base models (used / total),1/1
# DRF base models (used / total),2/2
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [26]:
# checking top performers

lb = aml.leaderboard
lb

model_id,rmse,mse,mae,rmsle,mean_residual_deviance
StackedEnsemble_BestOfFamily_4_AutoML_2_20250401_201013,23.1293,534.964,13.9832,nan,534.964
StackedEnsemble_BestOfFamily_3_AutoML_2_20250401_201013,23.1384,535.383,14.0269,nan,535.383
StackedEnsemble_AllModels_4_AutoML_2_20250401_201013,23.1385,535.388,13.9549,nan,535.388
StackedEnsemble_AllModels_1_AutoML_2_20250401_201013,23.1406,535.487,14.0419,nan,535.487
StackedEnsemble_AllModels_3_AutoML_2_20250401_201013,23.1407,535.492,13.971,nan,535.492
StackedEnsemble_BestOfFamily_2_AutoML_2_20250401_201013,23.1431,535.602,14.0509,nan,535.602
StackedEnsemble_AllModels_2_AutoML_2_20250401_201013,23.1434,535.619,14.0277,nan,535.619
GLM_1_AutoML_2_20250401_201013,23.1498,535.914,14.1269,nan,535.914
StackedEnsemble_BestOfFamily_1_AutoML_2_20250401_201013,23.1529,536.056,14.0765,nan,536.056
XGBoost_grid_1_AutoML_2_20250401_201013_model_33,23.2009,538.281,13.9541,nan,538.281


In [29]:
best_model = aml.leader  # Best model
performance = best_model.model_performance(valid)
print(performance)

ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 501.6339029516403
RMSE: 22.397185156881665
MAE: 13.076782927975117
RMSLE: NaN
Mean Residual Deviance: 501.6339029516403
R^2: 0.3445742769433672
Null degrees of freedom: 3455
Residual degrees of freedom: 3449
Null deviance: 2652086.115876017
Residual deviance: 1733646.768600869
AIC: 31312.66385441705
